In [1]:
import numpy as np
import pandas as pd
import re

import spacy
nlp = spacy.load('en_core_web_lg')

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [2]:
pos_list = [w.rstrip() for w in open(r"..\data\raw\pos.txt", 'r')]
neg_list = [w.rstrip() for w in open(r"..\data\raw\neg.txt", 'r')]

In [ ]:
def rhetoric_features_extractor(df):
  ## INÍCIO DA EXTRAÇÃO DAS META-VARIÁVEIS DE PATHOS ##
  def pathos_feature_extractor(df):
    def pos_neg_words(doc): 
      total_pos_w, total_neg_w = 0, 0 
      total_pos_jj, total_neg_jj = 0, 0
      total_pos_nn, total_neg_nn = 0, 0

      for token in doc:
        word, tag = token.lemma_, token.tag_

        if word.lower() in pos_list:
          total_pos_w += 1
          if tag.startswith('JJ'): total_pos_jj += 1
          elif tag.startswith('NN'): total_pos_nn += 1

        elif word.lower() in neg_list:
          total_neg_w += 1
          if tag.startswith('JJ'): total_neg_jj += 1
          elif tag.startswith('NN'): total_neg_nn += 1
      return pd.Series([total_pos_w, total_pos_jj, total_pos_nn, total_neg_w, total_neg_jj, total_neg_nn])

    def sentiment_analyzer(text):
      score = SentimentIntensityAnalyzer().polarity_scores(text)
      return pd.Series([score['pos'], score['neu'], score['neg'], score['compound']])

    df[['total_pos_w', 'total_pos_jj', 'total_pos_nn', 'total_neg_w', 'total_neg_jj', 'total_neg_nn']] = df['doc'].apply(pos_neg_words)
    df[['pos_sentiment_polarity', 'neu_sentiment_polarity', 'neg_sentiment_polarity', 'comp_sentiment_polarity']] = df['text'].apply(sentiment_analyzer)
    ## FIM DA EXTRAÇÃO DAS META-VARIÁVEIS DE PATHOS ## 

  ## INÍCIO DA EXTRAÇÃO DAS META-VARIÁVEIS DE ETHOS ##
  def ethos_feature_extractor(df):
    def pronoun_sent(doc):
      # pronoun sentences are sentences who have pronouns as subject or object
      for token in doc:
        sub_obj_pattern = r'^(nsubj(pass)?|p?obj)$'
        pronoun_pattern = r'^PRP.?$'
        if re.match(sub_obj_pattern, token.dep_):
          if re.match (pronoun_pattern, token.tag_):
            return 1
      return 0

    def future_tense(doc): # checks for nsubj > aux > verb
      for token in doc:
        if token.dep_ == 'nsubj' and token.head.pos_ == 'VERB':
          for child in token.head.children:
            if child.pos_ == 'AUX':
              return 1
      return 0

    def has_wh(doc): # probably will use this function for singling out grammar classes
      wh_regex = r'^(WDT|WP|WP$|WRB)$'
      for token in doc:
        if re.match(wh_regex, token.tag_):
          return 1
      return 0

    def subjecitivity(doc):
      wh_regex = r'^(JJ.?|RB.?)$'
      for token in doc:
        if re.match(wh_regex, token.tag_):
          return 1
      return 0

    def conditional(doc): #checks for subordinating conjunctions
      wh_regex = r'^SCONJ$'
      for token in doc:
        if re.match(wh_regex, token.pos_):
          return 1
      return 0

    # lexical variables
    df['pronoun_sent'] = df['doc'].apply(pronoun_sent)

    # sentence variables
    df['future_tense'] = df['doc'].apply(future_tense)

    # textual variables
    df['has_interrogation'] = df['text'].apply(lambda text: 1 if '?' in text else 0)
    df['has_wh'] = df['doc'].apply(has_wh)
    df['subjecitivity'] = df['doc'].apply(subjecitivity)
    df['conditional'] = df['doc'].apply(conditional)
  ## FIM DA EXTRAÇÃO DAS META-VARIÁVEIS DE ETHOS ##

  ## INÍCIO DA EXTRAÇÃO DAS META-VARIÁVEIS DE LOGOS ##
  def logos_feature_extractor(df):
    def numerical_nominal(doc): # for numerical-nominal texts
      for i in range (len(doc)-1):
        if doc[i].tag_ == 'CD' and re.match(r'NN(S|P|PS)?', doc[i + 1].tag_):
          return 1
      return 0

    def ner_recognizer(text, label):
      doc = nlp(text.lower())
      return len([ent for ent in doc.ents if ent.label_ == label])

    def has_complex_sent(sent_list): # sentential level
      # for finding complex sentences (subj + verb(root)+ clause)
      has_complex = 0
      sum_log_sent = 0
      for sent in sent_list:
        if len(sent) != 0: sum_log_sent += np.log(len(sent))
        doc = nlp(sent)
        root_verb, clause_mod, mark_count = 0, 0, 0
        for token in doc:
          # searches for a root verb
          sub_obj_pattern = r'^(nsubj(pass)?|p?obj)$' # for sentences without subject
          if re.match(sub_obj_pattern, token.dep_) and token.head.pos_ == 'VERB' and token.head.dep_ == 'ROOT':
          #if token.dep_ == 'nsubj' and token.head.pos_ == 'VERB' and token.head.dep_ == 'ROOT':
            root_verb += 1
            #checks for dependant clauses (adverbial or relative)
            for child in token.head.children:
              if child.dep_ in ['advcl', 'relcl']:
                clause_mod +=1
              # if not, checks if is a subordination conjugation
              elif child.dep_ == 'mark':
                mark_count += 1
        if(root_verb + clause_mod + mark_count) > 1: has_complex = 1
      return pd.Series([has_complex, sum_log_sent])

    def passive_speech(doc): # checks for passive speech (aux + verb past)
      for token in doc:
        if token.dep_ == 'nsubjpass' and token.head.tag_ =='VBN':
          for child in token.head.children:
            if child.pos_ == 'AUX': return 1
      return 0

    # lexical variables
    df['numerical_nominal'] = df['doc'].apply(numerical_nominal) # 918 entries

    df['organization_named'] = df['text'].apply(lambda text: ner_recognizer(text, 'ORG'))
    df['person_named'] = df['text'].apply(lambda text: ner_recognizer(text, 'PERSON'))

    # sentence variables
    df[['has_complex_sent', 'sum_log_sent']] = df['sent_tokens'].apply(has_complex_sent) # returns 1059 positive

    # textual variables
    pronouns = ['he', 'him', 'his', 'she', 'her', 'hers', 'they', 'their', 'them']

    df['third_p_count'] = df['doc'].apply(lambda doc: sum(1 for token in doc
                                                                if token.text.lower() in pronouns))
    df['has_passive'] = df['doc'].apply(passive_speech)
    df['has_period'] = df['text'].apply(lambda text: 1 if r'.' in text else 0)
    df['has_exclamation'] = df['text'].apply(lambda text: 1 if r'!' in text else 0)
  ## FIM DA EXTRAÇÃO DAS META-VARIÁVEIS DE LOGOS ##

  pathos_feature_extractor(df)
  ethos_feature_extractor(df)
  logos_feature_extractor(df)

In [ ]:
def bing_liu_features_extractor(df):
  def senty_analysis(doc):
    total_pos_w, total_neg_w = 0, 0 # Total de palavras positivas/negativas
    total_pos_jj, total_neg_jj = 0, 0 # total de adjetivos pos/neg
    total_pos_nn, total_neg_nn = 0, 0 # total de substantivos pos/neg
    total_pos_rb, total_neg_rb = 0, 0 # total de advérbios pos/neg
    total_pos_vb, total_neg_vb = 0, 0 # total de verbos pos/neg

    for token in doc:
      word, tag = token.lemma_, token.tag_
      if word in pos_list:
        total_pos_w += 1
        if tag.startswith('JJ'): total_pos_jj += 1  # Adjectives
        elif tag.startswith('NN'): total_pos_nn += 1  # Nouns
        elif tag.startswith('RB'): total_pos_rb += 1  # Adverbs
        elif tag.startswith('VB'): total_pos_vb += 1  # Verbs
      elif word in neg_list:
        total_neg_w += 1
        if tag.startswith('JJ'): total_neg_jj += 1  # Adjectives
        elif tag.startswith('NN'): total_neg_nn += 1  # Nouns
        elif tag.startswith('RB'): total_neg_rb += 1  # Adverbs
        elif tag.startswith('VB'): total_neg_vb += 1  # Verbs

    return pd.Series([total_pos_w, total_pos_jj, total_pos_nn, total_pos_rb, total_pos_vb,
                      total_neg_w, total_neg_jj, total_neg_nn, total_neg_rb, total_neg_vb])

  df[['total_pos_w', 'total_pos_jj', 'total_pos_nn', 'total_pos_rb', 'total_pos_vb',
      'total_neg_w', 'total_neg_jj', 'total_neg_nn', 'total_neg_rb', 'total_neg_vb']] = df.doc.apply(senty_analysis)
  
  # Extração das meta-variáveis MOL já realizadas

In [5]:
def leonardo_feature_extraction(df):
  # total of sentences
  df['total_of_sentences'] = df['sent_tokens'].apply(lambda s: len(s))

  # character mean per sentece
  df['char_mean_sentence'] = df['sent_tokens'].apply(lambda s:
                                             float(np.mean([len(sent)
                                             for sent in s])) if s else 0)

  # character variance per sentece
  df['char_variance_sentence'] = df['sent_tokens'].apply(lambda s:
                                             float(np.var([len(sent)
                                             for sent in s])) if s else 0)

  # total of characters per meme text
  df['total_char_text'] = df['text'].apply(lambda t: len(t) if t else 0)

  # character mean per word
  df['char_mean_word'] = df['doc'].apply(lambda doc:
                                             np.mean([len(token.text)
                                             for token in doc] if doc else 0))

  # character variance per word
  df['char_variance_word'] = df['doc'].apply(lambda doc:
                                             np.var([len(token.text)
                                             for token in doc] if doc else 0))

  # absolute frequency of all punctuation
  df['punctuation_frequency'] = df['doc'].apply(lambda doc:
                                             sum(1 for token in doc
                                                  if token.pos_ == 'PUNCT'))

  # absolute frequency of upper case characters
  df['upper_case_frequency'] = df['text'].apply(lambda text:
                                                sum(1 for letter in text if letter.isupper()))

  # lemmatized words proportion per token
  df['lemma_per_token'] = df['doc'].apply(lambda doc:
                                             (sum(1 for token in doc
                                                  if token.pos_ != 'PUNCT')
                                             / len(doc) if doc else 0) )

In [6]:
def carvalho_feature_extraction(df):
  def negation_attr(doc):
    negation_lexicon = ['no', 'not', 'never', 'nothing', 'none', 'nobody', 'nowhere', 'neither', 'nor',
        'barely', 'hardly', 'scarcely', 'rarely', 'seldom', 'without', 'lack',
        "don't", "doesn't", "didn't", "isn't", "aren't", "wasn't", "weren't",
        "haven't", "hasn't", "hadn't", "won't", "wouldn't", "shouldn't", "couldn't",
        "mightn't", "mustn't", "can't", "cannot"]
    
    text_lowered = doc.text.lower()

    tokens_punkt = re.findall(r"(\w+'\w+|\w+|[.,!?;:])", text_lowered) # captures words with apostrophes

    neg_word_count = 0

    punctuation = r'[.,!?;:]'

    negated_context = []
    current_split =[]
    neg = False

    for token in tokens_punkt:
      if token in negation_lexicon:
        neg = True
        neg_word_count += 1
      if neg:
        current_split.append(token)
        if token[-1] in punctuation:
            negated_context.append(current_split)
            current_split = []
            neg = False
        if doc.text.endswith(token):
            negated_context.append(current_split)

    return pd.Series([neg_word_count, len(negated_context)])


  df[['total_negation_words', 'total_negated_context']] = df['doc'].apply(negation_attr)

In [7]:
def feats_extractor(name=str, file_path = str):
    df_name = name
    df = pd.read_pickle(file_path)

    rhetoric_features_extractor(df)
    bing_liu_features_extractor(df)
    leonardo_feature_extraction(df)
    carvalho_feature_extraction(df)

    print(df.head())

    return df

In [8]:
train = feats_extractor('train', r"..\data\processed\train_cl.pkl")

      id                                               text  \
0  65635  THIS IS WHY YOU NEED A SHARPIE WITH YOU AT ALL...   
1  67927  GOOD NEWS! NAZANIN ZAGHARI-RATCLIFFE AND ANOOS...   
2  68031                            PAING PHYO MIN IS FREE!   
3  77490  Move your ships away! oooook Move your ships a...   
4  67641           WHEN YOU'RE THE FBI, THEY LET YOU DO IT.   

                                              labels  persuasion  \
0             [Black-and-white Fallacy/Dictatorship]           1   
1  [Loaded Language, Glittering generalities (Vir...           1   
2                                                 []           0   
3                                                 []           0   
4                       [Thought-terminating cliché]           1   

                                         sent_tokens  \
0  [THIS IS WHY YOU NEED A SHARPIE WITH YOU AT AL...   
1  [GOOD NEWS!, NAZANIN ZAGHARI-RATCLIFFE AND ANO...   
2                          [PAING PHYO MIN I

In [9]:
test = feats_extractor('test', r"..\data\processed\test_cl.pkl")

      id                                               text  \
0  63292     This is why we're free This is why we're safe    
1  70419  IF YOU SAY WE'RE IN THE MIDDLE OF A DEADLY PAN...   
2  63673  AFTER A YEAR OF ZERO EVIDENCE OF RUSSIAN COLLU...   
3  71297                                      MY WAY MY WAY   
4  66340  Putin signed a decree to exclude Lyman from Ru...   

                                              labels  persuasion  \
0                        [Causal Oversimplification]           1   
1  [Loaded Language, Name calling/Labeling, Black...           1   
2  [Smears, Misrepresentation of Someone's Positi...           1   
3                                                 []           0   
4                                  [Loaded Language]           1   

                                         sent_tokens  \
0    [This is why we're free This is why we're safe]   
1  [IF YOU SAY WE'RE IN THE MIDDLE OF A DEADLY PA...   
2  [AFTER A YEAR OF ZERO EVIDENCE OF RUSSIAN

exportation

In [10]:
atts = ['id', 'persuasion',
        # 'pathos', 'ethos', 'logos', 
        'hate_speech_terms_total', 'hate_speech_expressions_total',
       'total_hate_jj', 'total_hate_nn', 'total_hate_rb', 'total_hate_vb',
       'total_pos_w', 'total_pos_jj', 'total_pos_nn', 'total_neg_w',
       'total_neg_jj', 'total_neg_nn', 'pos_sentiment_polarity',
       'neu_sentiment_polarity', 'neg_sentiment_polarity',
       'comp_sentiment_polarity', 'pronoun_sent', 'future_tense',
       'has_interrogation', 'has_wh', 'subjecitivity', 'conditional',
       'numerical_nominal', 'organization_named', 'person_named',
       'has_complex_sent', 'sum_log_sent', 'third_p_count', 'has_passive',
       'has_period', 'has_exclamation', 'total_pos_rb', 'total_pos_vb',
       'total_neg_rb', 'total_neg_vb', 'total_of_sentences',
       'char_mean_sentence', 'char_variance_sentence', 'total_char_text',
       'char_mean_word', 'char_variance_word', 'punctuation_frequency',
       'upper_case_frequency', 'lemma_per_token', 'total_negation_words',
       'total_negated_context']

In [11]:
train = train.loc[:, atts]
test = test.loc[:, atts]

In [12]:
train.to_pickle(r"..\data\processed\train.pkl")
test.to_pickle(r"..\data\processed\test.pkl")

In [13]:
#train.to_csv(r"C:\Users\anabe\OneDrive\Documentos\R Analysis\train.csv")
#validation.to_csv(r"C:\Users\anabe\OneDrive\Documentos\R Analysis\validation.csv")
#test.to_csv(r"C:\Users\anabe\OneDrive\Documentos\R Analysis\test.csv")